# Actividad 05: Consultas analíticas

El warehouse está cargado. Ahora vamos a responder las 7 preguntas de negocio de Wesker.

> Las 7 preguntas están en el [brief de Wesker](../BRIEF.md).

In [ ]:
import duckdb

con = duckdb.connect(':memory:')

# Create sample tables with data so you can test your queries
con.execute('''
    CREATE TABLE experiment_fact (
        experiment_id INTEGER PRIMARY KEY,
        date_id INTEGER,
        laboratory_id INTEGER,
        researcher_id INTEGER,
        sample_id INTEGER,
        project_id INTEGER,
        processing_time_minutes INTEGER,
        sample_count INTEGER,
        experiment_cost DECIMAL(10,2),
        is_failed BOOLEAN
    );
''')
con.execute('''
    INSERT INTO experiment_fact VALUES
    (1, 20260115, 3, 14, 101, 1, 45, 12, 1500.00, false),
    (2, 20260120, 7, 22, 102, 2, 60, 8, 2200.00, false),
    (3, 20260201, 3, 30, 103, 1, 30, 15, 900.00, true),
    (4, 20260210, 9, 31, 104, 3, 90, 5, 3100.00, false),
    (5, 20260305, 3, 40, 105, 1, 55, 20, 1800.00, false),
    (6, 20260315, 7, 14, 106, 2, 40, 10, 1200.00, false),
    (7, 20260401, 3, 22, 107, 1, 75, 8, 2500.00, true),
    (8, 20260410, 9, 30, 108, 3, 50, 14, 1700.00, false),
    (9, 20250615, 3, 14, 109, 1, 35, 18, 1100.00, false),
    (10, 20250720, 7, 22, 110, 2, 65, 6, 2800.00, false);
''')

con.execute('''
    CREATE TABLE laboratory_dim (
        laboratory_id INTEGER PRIMARY KEY,
        laboratory_name VARCHAR,
        city VARCHAR,
        country VARCHAR,
        security_level INTEGER
    );
''')
con.execute('''
    INSERT INTO laboratory_dim VALUES
    (3, 'Arklay Research Lab', 'Ringtail City', 'USA', 5),
    (7, 'Tanuki BioPlant', 'Osaka', 'Japan', 3),
    (9, 'Spencer R&D Annex', 'Geneva', 'Switzerland', 4);
''')

con.execute('''
    CREATE TABLE researcher_dim (
        researcher_id INTEGER PRIMARY KEY,
        full_name VARCHAR,
        position VARCHAR,
        department VARCHAR,
        country VARCHAR
    );
''')
con.execute('''
    INSERT INTO researcher_dim VALUES
    (14, 'Rebecca Chambers', 'Junior Scientist', 'Bioresearch', 'USA'),
    (22, 'Marcus Aiken', 'Senior Scientist', 'Bioresearch', 'UK'),
    (30, 'Ada Wong', 'Senior Scientist', 'Advanced Research', 'China'),
    (31, 'Leon Kennedy', 'Field Analyst', 'Security', 'USA'),
    (40, 'Claire Redfield', 'Junior Scientist', 'Bioresearch', 'USA');
''')

con.execute('''
    CREATE TABLE sample_dim (
        sample_id INTEGER PRIMARY KEY,
        sample_type VARCHAR,
        risk_level VARCHAR,
        origin_country VARCHAR
    );
''')
con.execute('''
    INSERT INTO sample_dim VALUES
    (101, 'Blood', 'High', 'USA'),
    (102, 'Tissue', 'Medium', 'Japan'),
    (103, 'Saliva', 'Low', 'Spain'),
    (104, 'Urine', 'High', 'Germany'),
    (105, 'Blood', 'Medium', 'USA'),
    (106, 'Tissue', 'Low', 'Japan'),
    (107, 'Blood', 'High', 'USA'),
    (108, 'Saliva', 'Medium', 'Italy'),
    (109, 'Blood', 'Low', 'USA'),
    (110, 'Tissue', 'High', 'Japan');
''')

con.execute('''
    CREATE TABLE project_dim (
        project_id INTEGER PRIMARY KEY,
        project_name VARCHAR,
        category VARCHAR
    );
''')
con.execute('''
    INSERT INTO project_dim VALUES
    (1, 'Project Arklay', 'Virology'),
    (2, 'Project T-virus', 'Bioweapons'),
    (3, 'Project Nemesis', 'Genetics');
''')

con.execute('''
    CREATE TABLE date_dim (
        date_id INTEGER PRIMARY KEY,
        full_date DATE,
        year INTEGER,
        month INTEGER,
        day INTEGER,
        day_name VARCHAR,
        is_weekend BOOLEAN
    );
''')
con.execute('''
    INSERT INTO date_dim VALUES
    (20260115, '2026-01-15', 2026, 1, 15, 'Thursday', false),
    (20260120, '2026-01-20', 2026, 1, 20, 'Tuesday', false),
    (20260201, '2026-02-01', 2026, 2, 1, 'Sunday', true),
    (20260210, '2026-02-10', 2026, 2, 10, 'Tuesday', false),
    (20260305, '2026-03-05', 2026, 3, 5, 'Thursday', false),
    (20260315, '2026-03-15', 2026, 3, 15, 'Sunday', true),
    (20260401, '2026-04-01', 2026, 4, 1, 'Wednesday', false),
    (20260410, '2026-04-10', 2026, 4, 10, 'Friday', false),
    (20250615, '2025-06-15', 2025, 6, 15, 'Sunday', true),
    (20250720, '2025-07-20', 2025, 7, 20, 'Sunday', true);
''')

print('Sample warehouse loaded with 10 experiments across 3 labs.')
print('Tables: experiment_fact, laboratory_dim, researcher_dim, sample_dim, project_dim, date_dim')


## El modelo sobre el que vas a consultar

Este es el esquema estrella del warehouse. Cada consulta empieza en `experiment_fact` y une las dimensiones que necesite.

```mermaid
erDiagram
    DATE_DIM ||--o{ EXPERIMENT_FACT : FK-date
    LABORATORY_DIM ||--o{ EXPERIMENT_FACT : FK-lab
    RESEARCHER_DIM ||--o{ EXPERIMENT_FACT : FK-researcher
    SAMPLE_DIM ||--o{ EXPERIMENT_FACT : FK-sample
    PROJECT_DIM ||--o{ EXPERIMENT_FACT : FK-project
    EXPERIMENT_FACT {
        int experiment_id PK
        int date_id FK
        int laboratory_id FK
        int researcher_id FK
        int sample_id FK
        int project_id FK
        int processing_time_minutes
        int sample_count
        float experiment_cost
        bool is_failed
    }
    DATE_DIM {
        int date_id PK
        int year
        int month
        bool is_weekend
    }
    LABORATORY_DIM {
        int laboratory_id PK
        string laboratory_name
        string city
        string country
    }
    RESEARCHER_DIM {
        int researcher_id PK
        string full_name
        string position
        string department
    }
    SAMPLE_DIM {
        int sample_id PK
        string sample_type
        string risk_level
        string origin_country
    }
    PROJECT_DIM {
        int project_id PK
        string project_name
        string category
    }
```


### Qué dimensión necesita cada pregunta

Antes de escribir cada consulta, mira en este mapa qué tablas tienes que unir.

```mermaid
flowchart TB
    subgraph FECHA["date_dim"]
        P1["P1: coste por mes"]
        P4["P4: 2025 vs 2026"]
        P7["P7: laborable vs fin de semana"]
    end
    subgraph LAB["laboratory_dim"]
        P2["P2: tiempo medio de procesamiento"]
        P5["P5: laboratorio Arklay"]
    end
    subgraph MUE["sample_dim"]
        P3["P3: muestras de alto riesgo"]
    end
    subgraph SOLO["Solo experiment_fact"]
        P6["P6: coste por muestra"]
    end
```


### Pregunta 1: Costes mensuales de experimentos

*Ver enunciado en el [brief](../BRIEF.md).*

In [ ]:
# TODO: Escribe la consulta



### Pregunta 2: Tiempo medio de procesamiento por laboratorio

*Ver enunciado en el [brief](../BRIEF.md).*

In [ ]:
# TODO: Escribe la consulta



### Pregunta 3: Muestras de alto riesgo

*Ver enunciado en el [brief](../BRIEF.md).*

In [ ]:
# TODO: Escribe la consulta



### Pregunta 4: Comparativa anual (2025 vs 2026)

*Ver enunciado en el [brief](../BRIEF.md).*

In [ ]:
# TODO: Escribe la consulta



### Pregunta 5: Laboratorio Arklay

*Ver enunciado en el [brief](../BRIEF.md).*

In [ ]:
# TODO: Escribe la consulta



### Pregunta 6: Coste por muestra

*Ver enunciado en el [brief](../BRIEF.md).*

In [ ]:
# TODO: Escribe la consulta



### Pregunta 7: Días laborables vs fin de semana

*Ver enunciado en el [brief](../BRIEF.md).*

In [ ]:
# TODO: Escribe la consulta



### Bonus: Cubo OLAP con ROLLUP

> **Recuerda:** `ROLLUP` agrega subtotales a cada nivel de la jerarquía del cubo OLAP.

Crea un cubo OLAP con ROLLUP para analizar costes por laboratorio, investigador y nivel de riesgo.

### El cubo, nivel a nivel

`ROLLUP` va subiendo niveles de agregación: cada línea del resultado suma el nivel anterior como subtotal.

```mermaid
flowchart TB
    T["TOTAL<br/>todos los experimentos"] --> L["por laboratorio"]
    L --> R["por laboratorio + investigador"]
    R --> K["por laboratorio + investigador + riesgo"]
```


In [ ]:
# TODO: Escribe la consulta ROLLUP



---

## Cierre: Comparativa OLTP vs OLAP

> **Recuerda:** los sistemas **OLTP** ejecutan las transacciones del día a día (consultas simples, pocas filas, optimizado para escritura); los sistemas **OLAP** analizan (consultas complejas, muchas filas, optimizado para lectura).

Ya has vivido ambos mundos: OLTP en la actividad 01 y OLAP aquí. Rellena `oltp_olap_comparison` con lo que has experimentado.

### OLTP y OLAP, cara a cara

Mira las diferencias y rellena la comparación de abajo.

```mermaid
flowchart TB
    subgraph OLTP["OLTP: el día a día"]
        T1["INSERT / UPDATE / DELETE"]
        T2["Búsqueda por PK"]
        T3["Muchas transacciones cortas"]
    end
    subgraph OLAP["OLAP: el análisis"]
        A1["SELECT con GROUP BY"]
        A2["JOIN a dimensiones"]
        A3["Pocas consultas, muchos datos"]
    end
```


In [ ]:
# TODO: Rellena la comparación

oltp_olap_comparison = {
    'oltp_query_type': '',        # Simple o Compleja?
    'olap_query_type': '',        # ?
    'oltp_rows_affected': '',     # Pocas o Muchas?
    'olap_rows_affected': '',     # ?
    'oltp_optimized_for': '',     # Escrituras o Lecturas?
    'olap_optimized_for': '',     # ?
    'oltp_use_case': '',          # Ejemplo de uso
    'olap_use_case': '',          # ?
}


## Guarda tu trabajo

1. Copia tus consultas SQL en `scripts/queries.sql`.
2. Copia tu comparativa (`oltp_olap_comparison`) en `scripts/oltp_olap.py`.

```bash
pytest tests/ -v
```

### Siguiente: Actividad 06: Slowly Changing Dimensions (módulo final)
